# Week 8 · QLoRA 微调 Qwen 1.5B

> **本周一句话**:4-bit 量化加载 Qwen 2.5-1.5B-Instruct,只训 4.3M LoRA 参数(占总参 0.28%),在 T4 上 30 分钟教会它写古诗 —— 对照自己的 25M 模型,直观感受"商业级 vs 自己造"的差距。

这是项目的收尾周。前 7 周你从零造了一个能写诗的小模型;本周用同样的数据微调一个"巨人",体感到 scale 和预训练数据的力量。

## 0. 本周目标

| 维度 | 自家 v0.10 | Qwen + LoRA |
|---|---|---|
| 基座 | 从零训 | Qwen2.5-1.5B-Instruct(已预训 7T+ tokens) |
| 参数 | 25 M 全部可训 | 1.5 B 总参 / **4.3 M 可训(LoRA only)** |
| 显存 | 训练 ~1 GB | 训练 ~6 GB(4-bit base + fp32 LoRA) |
| 数据 | 我们的唐宋诗词 | 同样,但转 Qwen chat 格式 |
| 单条生成质量 | 押韵 OK,偶尔出金句 | 流畅、押韵、能写英→中、能写现代题材 |
| 训练时间(T4)| 0.5 小时(SFT) | v1: 15 min,v2 继续: 30 min |

产出:`checkpoints/qwen_lora_v1/` 和 `qwen_lora_v2/`,各约 40 MB(只是 LoRA 补丁,不是完整模型)。

## 1. 前置知识

**必备**:
- Week 6 SFT 跑完(理解 prompt mask、chat 格式)
- 知道什么是 fp16 / fp32

**这周第一次遇到**:
- NF4 4-bit 量化(`bitsandbytes`)
- LoRA 的数学:`W' = W + (α/r) · BA`
- `peft` 库(LoraConfig / get_peft_model / PeftModel)
- `trl.SFTTrainer`(HuggingFace 的"开箱即用"训练器)
- gradient_checkpointing(用计算换显存)
- paged_adamw_8bit

## 2. 核心概念

### 2.1 NF4 4-bit 量化(QLoRA 的 Q)

Qwen 1.5B 原始 fp16 权重约 3 GB。NF4 把每个权重压成 4-bit,理论 4× 压缩 → ~750 MB。加上 scale / zero_point 等元数据,实际约 1.2 GB。

**NF4(NormalFloat 4-bit)的设计**:
- 神经网络权重大致正态分布
- 普通 INT4 在正态分布上分辨率浪费(尾部值少但占 ID)
- NF4 = 把 16 个量化级别按正态分布的 16 等分位点排布,信息论意义上最优

**double_quant**:量化的 scale 本身也再量化一次,再省 10%。

**关键限制**:4-bit 权重**不能直接训** —— 量化函数不可导。所以 QLoRA 用 LoRA 补丁来"绕开"。

### 2.2 LoRA 的数学(QLoRA 的 LoRA)

对一个原始 `Linear: W ∈ ℝ^{d×k}`(比如 Qwen 的 q_proj),LoRA 加一个低秩补丁:

$$W' = W + \frac{\alpha}{r} \cdot B A$$

where:
- `A ∈ ℝ^{r×k}`,标准正态初始化
- `B ∈ ℝ^{d×r}`,**初始化为 0**(关键:开始时 BA=0,W' = W,模型行为完全等同 base)
- `r = 16`(秩),`α = 32`(scaling = α/r = 2)

**参数量对比**:
- 原 `W`: `d × k`(假设 1024×1024 = 1M)
- LoRA `A + B`: `r × k + d × r` = 16×1024 + 1024×16 = 32K  → **省 30×**

推理时可以选择 merge: `W ← W + (α/r)BA`,变回普通 Linear。我们的 `generate.py` 不 merge(用 PeftModel 包装),省事但稍慢。

**target_modules = ["q_proj", "k_proj", "v_proj", "o_proj"]**:只贴在 attention 的 Q/K/V/O 上,不贴 MLP。LoRA 论文实验:Attention 是泛化的关键,LoRA 在 Attention 上的性价比最高。

### 2.3 FP32 LoRA + 4-bit base 的精度棋牌

量化模型直接训会出问题。**正确做法**:

```python
model = prepare_model_for_kbit_training(model)   # peft 准备:启梯度检查点,fp16 norm 等
model = get_peft_model(model, lora_config)       # 加 LoRA 补丁

# 关键修复:LoRA 补丁手动转 FP32
for _, p in model.named_parameters():
    if p.requires_grad:
        p.data = p.data.to(torch.float32)

# SFTConfig 必须关 fp16 / bf16
training_args = SFTConfig(fp16=False, bf16=False, ...)
```

**为什么不能开 AMP**?
- base 权重已经是 4-bit(bnb_4bit_compute_dtype=fp16,反量化到 fp16 算 forward)
- LoRA 补丁如果再走 fp16 → 梯度数值范围窄 → 训不动
- 让 LoRA 走 fp32,与 4-bit base 配合 —— 这是 QLoRA 原论文的标准做法

**SFTConfig 开 fp16=True 是最常见的错误**(因为大家训普通模型时都开)。我们 cell 9 就是这么错的,cell 10 才修对。

### 2.4 gradient_checkpointing + paged_adamw_8bit 进一步压显存

**gradient_checkpointing**:不缓存中间 activation,反向时重新算一遍。
- 显存:省 30-50%(activation 占大头)
- 速度:慢 20-30%(多算一次 forward)
- 对 T4 这种小显存卡,稳赢的交易

**paged_adamw_8bit**:把 Adam 的 m/v 动量从 fp32 (4 bytes) 压到 8-bit (1 byte)。
- 显存:m+v 状态省 4×
- 精度:8-bit 动量基本不影响最终质量(bitsandbytes 论文验证)

加上这两个,Qwen 1.5B + LoRA + batch=2 在 T4 上稳跑 6 GB。

### 2.5 v1 vs v2 训练策略

**v1**:朴素采样 2000 条样本,2 epoch,lr=2e-4
- 风格分布天然不均(七绝最多)
- 模型会偏向七绝

**v2**:均衡采样每种风格 1500 条 = 6000 条,3 epoch,lr=1e-4(基于 v1 继续训)
- 五绝 / 七绝 / 五律 / 七律 各得到充分训练
- lr 减半防止破坏 v1 学到的格律

为什么不一步到位训 v2?教学价值:**先看到失败的偏好,再修**,理解平衡数据的必要性。

## 3. 代码地图

| 文件 | 行数 | 干什么 |
|---|---|---|
| `qlora/load_qwen.py` | 45 | NF4 4-bit 加载 Qwen,~1.2 GB 显存 |
| `qlora/zero_shot_baseline.py` | 75 | 微调前对照(6 题写诗 + 4 题英→中) |
| `qlora/prepare_data.py` | 135 | 把 Week 6 SFT 数据转 Qwen chat 格式,支持 `--balanced` |
| `qlora/train.py` | 160 | LoRA 训练,带 `--v2` 续训均衡数据 |
| `qlora/generate.py` | 120 | 加载 LoRA + 生成,带 peft torchao patch |

依赖比前 7 周重:`transformers + accelerate + bitsandbytes + peft + trl + datasets`,合计 ~2 GB 安装。

## 4. 动手做

**先装 qlora 额外依赖**:

```powershell
uv sync --extra qlora
```

依赖装好后,激活 venv,按顺序跑:

In [ ]:
import subprocess

# 1. 看微调前 Qwen 写诗水平(基线,~5 分钟首次下载模型)
subprocess.run(["python", "../qlora/zero_shot_baseline.py"], check=True)
# 预期: Qwen 已经能写出像样的古诗,但格律偶尔出错
#       英→中也能凑合写,但风格不够"唐诗味"

In [ ]:
# 2. 准备 v1 数据(秒级)
subprocess.run(["python", "../qlora/prepare_data.py"], check=True)
# 预期: train=1900, val=100, Qwen chat 格式

In [ ]:
# 3. v1 LoRA 训练(~15 分钟 T4)
subprocess.run(["python", "../qlora/train.py"], check=True)
# 预期:
#   "可训参数: 4.30M / 总 1.54B (0.279%)"
#   ~475 步训完
#   eval_loss 从 1.5 降到 0.8 左右

In [ ]:
# 4. 准备 v2 均衡数据
subprocess.run(["python", "../qlora/prepare_data.py", "--balanced"], check=True)
# 预期: train=5820, val=180, 每种风格 1500 条

In [ ]:
# 5. v2 继续训(基于 v1 LoRA,~30 分钟)
subprocess.run(["python", "../qlora/train.py", "--v2"], check=True)
# 预期: lr=1e-4 比 v1 的 2e-4 小一倍,3 epoch ~2200 步

In [ ]:
# 6. 看微调后效果
subprocess.run(["python", "../qlora/generate.py", "--battery"], check=True)
# 8 道题套件,对照前面 v0.9 / v0.10 写的同样题目

In [ ]:
# 7. 记忆测试: 看模型是否在背训练集
subprocess.run(["python", "../qlora/generate.py", "--memorize"], check=True)
# 续写《静夜思》前两句。如果一字不差输出"举头望明月,低头思故乡"
# = 模型在背诵训练集,过拟合了

## 5. 自测题

**A. 量化**
- A1 NF4 vs INT4 vs FP4 三种 4-bit 量化各自的特点?为什么 NF4 最适合权重?
- A2 double_quant 把 scale 也量化,精度损失明显吗?
- A3 量化后的模型推理慢还是快?为什么?

**B. LoRA**
- B1 r=16 vs r=8 vs r=64 各有什么取舍?
- B2 α=2r 的经验从哪来?如果 α=r 会怎样?
- B3 为什么只贴 q/k/v/o,不贴 MLP?如果全部贴呢?
- B4 LoRA 的 B 初始化为 0 是关键设计,如果 B 初始化为随机正态会怎样?

**C. QLoRA 精度组合**
- C1 base 4-bit、LoRA fp32、forward 算什么精度?
- C2 SFTConfig fp16=True + LoRA fp32 会怎样?(实际试过会 NaN)
- C3 gradient_checkpointing 慢 20-30% 但显存省一半,什么时候不值得开?

**D. 训练**
- D1 paged_adamw_8bit 把动量压成 8-bit,会影响最终质量吗?哪些场景下风险大?
- D2 v2 用 lr=1e-4(v1 的一半)续训,如果用 2e-4 会怎样?
- D3 LoRA 只占总参 0.28%,但学到的"指令能力"为什么够用?

## 6. 容易踩的坑

**坑 1:`SFTConfig(fp16=True)` + LoRA → NaN**

cell 9 的朴素配置(没明确处理精度)会让 GradScaler 和 4-bit 量化打架,loss 几步内变 NaN。**修复**: `fp16=False, bf16=False`,LoRA 手动转 fp32。

**坑 2:bitsandbytes 在 Windows 上装不上**

bnb 0.43+ 官方支持 Windows,但要求 CUDA 11.7+。如果你 PyTorch 是 cu128,bnb 应该自动 OK。装不上的话:
- 试 `pip install bitsandbytes-windows`(社区 fork)
- 或者 Week 8 整个搬 Colab 跑

**坑 3:`peft` 加载 LoRA 时报 `torchao` 错**

peft 0.12+ 启动时会探测 torchao,某些环境探测失败。我们的 `qlora/generate.py:_patch_peft_torchao` monkey-patch 关掉这个探测。如果你看到 `is_torchao_available` 报错,确认 patch 被调用。

**坑 4:Qwen tokenizer 用 chat_template 时漏掉 `add_generation_prompt=True`**

训练时:`add_generation_prompt=False`(完整对话)
推理时:`add_generation_prompt=True`(到 `<|im_start|>assistant\n` 截断,让模型从这里开始生成)

漏了 → 推理时模型可能"补全 user 输入"而不是回答。

**坑 5:保存 LoRA 后误以为是完整模型,删了 Qwen base → 推理时 base 重新下载 3 GB**

LoRA 保存的只是 ~40 MB 的补丁,推理时必须配合 base 一起加载。HuggingFace cache 默认在 `~/.cache/huggingface/`,清这个会触发重新下载。

## 7. 项目结业

现在你应该:
- ☑ `checkpoints/qwen_lora_v1/` 和 `v2/` 都存在
- ☑ `--battery` 跑出来 Qwen LoRA 写的诗明显比自家 v0.10 更流畅
- ☑ 能解释 QLoRA = "4-bit base + LoRA fp32 + 关 AMP" 这三件套
- ☑ 能解释为什么 0.28% 的可训参数能改变模型行为(LoRA 的低秩假设)

## 8 周回顾

| Week | 学到什么 |
|---|---|
| 1 | 数据 pipeline 的工程纪律 |
| 2 | Transformer 全套组件,Bigram → 6 层 GPT |
| 3 | LR schedule、AMP、grad clip、wd —— 工业级训练循环 |
| 4 | Checkpoint 系统、断点续训、TensorBoard 监控 |
| 5 | 数据 + 模型双扩,scaling law 的初步体感 |
| 6 | SFT、特殊 token、prompt mask、vocab 扩展权重迁移 |
| 7 | DPO 数学、自动评分、ref_model 概念 —— 不用 RLHF 也能对齐 |
| 8 | QLoRA = 4-bit + LoRA,在小显存上微调真实大模型 |

你现在掌握的工具栈,**和 2024 年开源 LLM 训练社区的主流栈差不多**。再往下走的方向:
- **分布式训练**:多卡 DDP / FSDP / DeepSpeed(数据 ~100B 起步)
- **MoE**:Mixtral 类型的稀疏激活
- **Long context**:RoPE / YaRN 把 context 推到 32K+
- **更高级对齐**:GRPO(DeepSeek)、KTO、IPO 等 DPO 变种
- **推理优化**:vLLM / PagedAttention / 量化推理

但所有这些都建立在你前 8 周打的基础上。**收工**。